In [17]:
# !pip install qiskit

In [18]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import UnitaryGate

from qiskit.primitives import StatevectorSampler
import numpy as np

In [19]:
shots=1024
qubits=3
marked_state = 0b111  # target state as integer (|111> = 7)

In [20]:
def measure(qc: QuantumCircuit):
    for i in range(qubits):
        qc.measure(i, i)

    sampler = StatevectorSampler()
    pm = generate_preset_pass_manager(optimization_level=1)

    isa = pm.run(qc)
    job = sampler.run([isa], shots=shots)
    pub = job.result()[0]
    return pub.data.c.get_counts()

def inversion_about_mean(qc: QuantumCircuit):
    # Inversion about mean circuit, derivation can be found in exercise sheet:
    # H^{⊗3} X^{⊗3} (X1 Z1 X1 Z1) CCZ_{1,2,3} X^{⊗3} H^{⊗3}

    qubits_list = list(range(qubits))

    qc.h(qubits_list)
    qc.x(qubits_list)

    qc.ccz(0, 1, 2)
    # - sign change
    for _ in range(2):
        qc.z(0)
        qc.x(0)

    qc.barrier()
    qc.x(qubits_list)
    qc.h(qubits_list)

def oracle(qc: QuantumCircuit, n: int, marked: int):
    """General phase oracle for n qubits.
    Constructs the diagonal unitary  I - 2|marked><marked|
    which flips the phase of the marked state and leaves all others unchanged.
    """
    N = 2**n
    diag = np.ones(N)
    diag[marked] = -1
    U = np.diag(diag)

    qc.barrier()
    qc.append(UnitaryGate(U, label=f"Oracle |{marked:0{n}b}>"), range(n))
    qc.barrier()

def construct_grover(qubits:int, k: int) -> QuantumCircuit:
    qc = QuantumCircuit(qubits, qubits)

    for i in range(qubits):
        qc.h(i)

    for i in range(k):
        oracle(qc, n=qubits, marked=marked_state)
        inversion_about_mean(qc)
    return qc

In [21]:
# We test with applying Grover operator G k times and measure success probability
for k in range(1, 11):
    qc = construct_grover(qubits, k)
    counts = measure(qc)

    success_counts = counts['111']
    probability = success_counts / shots

    theta = np.arcsin(np.sqrt(1/2**qubits))
    prediction = np.sin((2*k+1)*theta)**2
    print(f"k={k:2d}  success state probability: {probability:.5f}   theoretical prediction: {prediction:.5f}")

k= 1  success state probability: 0.78223   theoretical prediction: 0.78125
k= 2  success state probability: 0.95117   theoretical prediction: 0.94531
k= 3  success state probability: 0.37988   theoretical prediction: 0.33008
k= 4  success state probability: 0.01660   theoretical prediction: 0.01221
k= 5  success state probability: 0.54980   theoretical prediction: 0.54797
k= 6  success state probability: 1.00000   theoretical prediction: 0.99979
k= 7  success state probability: 0.58203   theoretical prediction: 0.57697
k= 8  success state probability: 0.02734   theoretical prediction: 0.01946
k= 9  success state probability: 0.30371   theoretical prediction: 0.30289
k=10  success state probability: 0.92871   theoretical prediction: 0.93127
